# 훈련 사진 0–9 라벨 분류
Runtime → Run all. T4 GPU를 사용하세요. Google Drive 연결을 허용하세요. 기존 도시 분류 훈련 명령은 실행하지 않습니다.
훈련 JPG만 압축 해제합니다. 공식 val의 파일 내용은 열거나 압축 해제하지 않습니다. 원본 ZIP 다운로드에는 전체 자료가 포함됩니다.
결과: from_colab_photo_labels.zip. 사진 이동·삭제, 선택 목록 변경 없음. 특징 캐시는 Drive에 보존됩니다.

In [ ]:
import subprocess
if subprocess.run(['nvidia-smi'], capture_output=True).returncode != 0:
    raise RuntimeError('Runtime → Change runtime type → T4 GPU. GPU 한도 초과면 다음 날 다시 실행하세요.')
subprocess.run(['pip', 'install', '-q', 'torch==2.6.0', 'torchvision==0.21.0', 'numpy>=1.26,<2.3', 'pillow>=10.0'], check=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
from pathlib import Path
repo = 'https://github.com/downtownwonn/mas1004-assignment2.git'
project = Path('/content/drive/MyDrive/mas1004/assignment2_photo_labels')
project.parent.mkdir(parents=True, exist_ok=True)
os.environ['GIT_TERMINAL_PROMPT'] = '0'
def git(*args):
    return subprocess.run(['git', *args], check=True, text=True, capture_output=True).stdout.strip()
git('config', '--global', '--add', 'safe.directory', str(project))
if not project.exists():
    git('clone', repo, str(project))
elif git('-C', str(project), 'remote', 'get-url', 'origin') != repo:
    raise RuntimeError('Existing folder belongs to a different repository; stopping without changes.')
git('-C', str(project), 'pull', '--ff-only', 'origin', 'main')
print(git('-C', str(project), 'log', '-1', '--oneline'))
os.chdir(project)

In [ ]:
import sys
from getpass import getpass
archive = '/content/drive/MyDrive/mas1004/starter.zip'
env = os.environ.copy()
if not Path(archive).exists():
    env['MAS1004_STARTER_LINK'] = getpass('Cyber Campus starter photos link (not saved): ')
try:
    subprocess.run([sys.executable, '-u', 'src/colab_photo_labels.py', '--archive', archive], check=True, env=env)
finally:
    env.pop('MAS1004_STARTER_LINK', None)

In [ ]:
from google.colab import files
files.download('from_colab_photo_labels.zip')